# 20 — Relational Data Merging, Joining, and Concatenation Mastery
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for combining datasets, executing relational joins, preventing Cartesian explosions, and enforcing schema contracts in Pandas.*

---

## 📌 Executive Summary & Interview Expectations
In data engineering and data science technical screens, **combining data** is the single most tested topic after aggregations. Interviewers expect you to know far more than basic syntax; they evaluate whether you understand **relational integrity, join cardinality, and performance implications**:
1. **Concatenation vs Merging**: Stacking schemas (`pd.concat`) vs key-based relational alignment (`pd.merge`).
2. **Join Mechanics & Set Theory**: Exact behavior of `inner`, `left`, `right`, `outer`, and `cross` joins.
3. **The Cartesian Explosion Hazard**: What happens when duplicate keys exist on both sides ($M \times N$ row multiplication) and how to protect pipelines with `validate={'1:1', '1:m', 'm:1'}`.
4. **Anti-Joins & Data Reconciliation**: Using `indicator=True` (`_merge`) to execute SQL `NOT IN` / `NOT EXISTS` checks and audit missing records.
5. **Multi-Table Star-Schema Joins**: Joining transactional tables with dimension/lookup tables via column-to-index mappings.

## 1. Environment Setup & Data Ingestion

We load two real-world relational systems:
1. **Meetup Groups System**: `groups1`, `groups2`, `categories`, `cities`.
2. **Restaurant Sales System**: `week_1_sales`, `week_2_sales`, `customers_occupation`, `foods`.

In [1]:
import numpy as np
import pandas as pd

# Load Meetup datasets
groups1 = pd.read_csv("groups1.csv")
groups2 = pd.read_csv("groups2.csv")
categories = pd.read_csv("categories.csv")
cities = pd.read_csv("cities.csv", dtype={"zip": "string"})

# Load Restaurant datasets
week1 = pd.read_csv("week_1_sales.csv")
week2 = pd.read_csv("week_2_sales.csv")
customers = pd.read_csv("customers_occupation.csv", index_col="ID")
foods = pd.read_csv("foods.csv", index_col="Food ID")

print(f"Meetup Groups 1: {groups1.shape}, Groups 2: {groups2.shape}")
print(f"Week 1 Sales: {week1.shape}, Week 2 Sales: {week2.shape}")
print(f"Customers Dim: {customers.shape}, Foods Dim: {foods.shape}")

Meetup Groups 1: (7999, 4), Groups 2: (8331, 4)
Week 1 Sales: (250, 2), Week 2 Sales: (250, 2)
Customers Dim: (1000, 5), Foods Dim: (10, 2)


## 2. Concatenating Datasets (`pd.concat`)

### 💡 Interview Tip: Index Duplication on Concat
- By default, `pd.concat([df1, df2])` preserves the original row indices of each DataFrame!
- If both have index 0..N, the concatenated result will have **duplicate index labels**, making subsequent `.loc[0]` return multiple rows!
- **Best Practice**: Use `ignore_index=True` unless you explicitly want a hierarchical index via `keys=['G1', 'G2']`.

In [2]:
# Stacking two groups vertically
print("Length G1:", len(groups1), "| Length G2:", len(groups2), "| Sum:", len(groups1) + len(groups2))

# 1. Preserving provenance with hierarchical MultiIndex
groups_multi = pd.concat([groups1, groups2], keys=["G1", "G2"])
print("\nHierarchical Concat Sample:")
display(groups_multi.head(3))

# 2. Production Standard: Clean continuous RangeIndex
groups = pd.concat([groups1, groups2], ignore_index=True)
print(f"\nUnified Groups Shape: {groups.shape}, Index Is Monotonic: {groups.index.is_monotonic_increasing}")
groups.head(3)

Length G1: 7999 | Length G2: 8331 | Sum: 16330

Hierarchical Concat Sample:


group_id                       name  category_id  city_id
G1 0      6388     Alternative Health NYC           14    10001
   1      6510  Alternative Energy Meetup            4    10001
   2      8458          NYC Animal Rights           26    10001


Unified Groups Shape: (16330, 4), Index Is Monotonic: True


,group_id,name,category_id,city_id
0,6388,Alternative Health NYC,14,10001
1,6510,Alternative Energy Meetup,4,10001
2,8458,NYC Animal Rights,26,10001


### Handling Disjoint Columns & Horizontal Concatenation
When DataFrames have mismatched columns:
- `join='outer'` (default): Union of all columns, filling non-overlapping cells with `NaN`.
- `join='inner'`: Intersection of columns (only keeps columns present in both).
- `axis=1`: Stacks columns side-by-side (aligning on index).

In [3]:
sports_A = pd.DataFrame(
    [["New England Patriots", "Houston Astros"], ["Philadelphia Eagles", "Boston Red Sox"]],
    columns=["Football", "Baseball"]
)
sports_B = pd.DataFrame(
    [["New England Patriots", "St. Louis Blues"], ["Kansas City Chiefs", "Tampa Bay Lightning"]],
    columns=["Football", "Hockey"]
)

print("Outer Concat (Union of Columns):")
display(pd.concat([sports_A, sports_B], ignore_index=True))

print("Inner Concat (Intersection of Columns):")
display(pd.concat([sports_A, sports_B], join="inner", ignore_index=True))

Outer Concat (Union of Columns):


,Football,Baseball,Hockey
0,New England Patriots,Houston Astros,NaN
1,Philadelphia Eagles,Boston Red Sox,NaN
2,New England Patriots,NaN,St. Louis Blues
3,Kansas City Chiefs,NaN,Tampa Bay Lightning


Inner Concat (Intersection of Columns):


,Football
0,New England Patriots
1,Philadelphia Eagles
2,New England Patriots
3,Kansas City Chiefs


## 3. Relational Joins: Left, Inner, and Outer

### Relational Join Matrix
| Join Type | `how` | Description | Unmatched Left Rows | Unmatched Right Rows |
| :--- | :--- | :--- | :--- | :--- |
| **Inner** | `'inner'` | Intersection of keys | Dropped | Dropped |
| **Left** | `'left'` | Keep all left rows | Preserved (fills Right with `NaN`) | Dropped |
| **Right** | `'right'` | Keep all right rows | Dropped | Preserved (fills Left with `NaN`) |
| **Outer** | `'outer'` | Full union of keys | Preserved | Preserved |
| **Cross** | `'cross'` | Cartesian product ($M \times N$) | N/A | N/A |

In [4]:
# 1. Left Join: Retain all groups, attach category names
groups_with_cats = groups.merge(categories, how="left", on="category_id")
print("Left Join Sample (Groups + Categories):")
display(groups_with_cats.head(3))

# 2. Inner Join: Only groups that match a known category
inner_joined = groups.merge(categories, how="inner", on="category_id")
print(f"Total Groups: {len(groups)} | Matched Inner Groups: {len(inner_joined)}")

Left Join Sample (Groups + Categories):


,group_id,name,category_id,city_id,category_name
0,6388,Alternative Health NYC,14,10001,Health & Wellbeing
1,6510,Alternative Energy Meetup,4,10001,Community & Environment
2,8458,NYC Animal Rights,26,10001,NaN


Total Groups: 16330 | Matched Inner Groups: 8037


## 4. Outer Joins, `indicator=True`, & The SQL Anti-Join

### 💡 Top Interview Question: "How do you do an Anti-Join in Pandas?"
SQL engineers frequently write `SELECT * FROM A WHERE key NOT IN (SELECT key FROM B)`.
In Pandas, there is no `how='anti'`.
**Idiomatic Solution**:
1. Perform a `merge(..., how='left' or 'outer', indicator=True)`
2. Filter where `_merge == 'left_only'` (or `'right_only'`)
3. Drop the `_merge` column!

In [5]:
# Outer join between groups and cities with indicator
outer_groups_cities = groups.merge(
    cities,
    how="outer",
    left_on="city_id",
    right_on="id",
    indicator=True
)

print("Indicator Distribution:")
display(outer_groups_cities["_merge"].value_counts())

# Anti-Join Drill: Cities that have NO Meetup groups registered
cities_without_groups = outer_groups_cities[
    outer_groups_cities["_merge"] == "right_only"
][["id", "city", "state", "zip"]].drop_duplicates()

print(f"\nNumber of Cities without any Meetup Groups: {len(cities_without_groups)}")
display(cities_without_groups.head(4))

Indicator Distribution:


_merge
both          16330
right_only        4
left_only         0
Name: count, dtype: int64


Number of Cities without any Meetup Groups: 4


,id,city,state,zip
8576,13417,New York Mills,NY,13417
8577,46312,East Chicago,IN,46312
8578,56567,New York Mills,MN,56567
16333,95712,Chicago Park,CA,95712


## 5. Column-to-Index Merging & Multi-Table Star Schemas

When lookup/dimension tables use their primary key as their DataFrame Index (e.g. `customers` indexed by `ID`, `foods` indexed by `Food ID`), use:
- `left_on='foreign_key'` and `right_index=True`.

In [6]:
# Merging Week 1 Transactions with Customers dimension
sales_with_customer = week1.merge(
    customers,
    how="left",
    left_on="Customer ID",
    right_index=True
)

# Merging with Foods dimension
complete_transactions = sales_with_customer.merge(
    foods,
    how="left",
    left_on="Food ID",
    right_index=True
)

print("Enriched Transaction (Transactions -> Customers -> Foods):")
display(complete_transactions.head(4))

Enriched Transaction (Transactions -> Customers -> Foods):


,Customer ID,Food ID,First Name,Last Name,Gender,Company,Occupation,Food Item,Price
0,537,9,Cheryl,Carroll,Female,Zoombeat,Registered Nurse,Donut,0.99
1,97,4,Amanda,Watkins,Female,Ozu,Account Coordinator,Quesadilla,4.25
2,658,1,Patrick,Webb,Male,Browsebug,Community Outreach Specialist,Sushi,3.99
3,202,2,Louis,Campbell,Male,Rhynoodle,Account Representative III,Burrito,9.99


## 6. Join Cardinality & The `validate` Parameter

### 🚨 Production Gotcha: Cartesian Multiplication Disasters
If a customer makes 3 purchases in Week 1 and 4 purchases in Week 2, merging on `Customer ID` generates **$3 \times 4 = 12$ rows** for that customer!
If you meant to join on a unique primary key, duplicate keys will silently explode row counts and distort revenue!

### The Defense: `validate`
Use the `validate` argument in `pd.merge()`:
- `validate='one_to_one'` (`'1:1'`): Checks if join keys are unique in both left and right datasets.
- `validate='one_to_many'` (`'1:m'`): Checks if join keys are unique in left dataset.
- `validate='many_to_one'` (`'m:1'`): Checks if join keys are unique in right dataset.
- `validate='many_to_many'` (`'m:m'`): Allows duplicate keys on both sides (default).
If the condition is violated, Pandas raises a **`MergeError`** immediately!

In [7]:
# Testing join validation
try:
    # We expect customer dimension to have unique IDs (many-to-one)
    week1.merge(customers, how="left", left_on="Customer ID", right_index=True, validate="many_to_one")
    print("Validation 'many_to_one' passed successfully! Customer IDs are unique.")
except pd.errors.MergeError as e:
    print(f"Validation failed: {e}")

# If we mistakenly assumed customer purchases between week1 and week2 were 1-to-1:
try:
    week1.merge(week2, how="inner", on="Customer ID", validate="one_to_one")
except pd.errors.MergeError as e:
    print(f"Caught expected MergeError on duplicate transaction keys:\n  -> {e}")

Validation 'many_to_one' passed successfully! Customer IDs are unique.
Caught expected MergeError on duplicate transaction keys:
  -> Merge keys are not unique in either left or right dataset; not a one-to-one merge.
Duplicates in left:
  Customer ID
         155
         304
         772
         504
          71 ...
Duplicates in right:
  Customer ID
         761
         343
         253
         496
         198 ...


---
## 🎯 7. Technical Interview Corner: Tricky Questions & Drills

### Q1: What is the difference between `df.merge()` and `df.join()`?
**Answer**:
1. **`df.merge()`**:
   - Universal relational join function.
   - Merges on **columns by default** (or columns to indexes, or indexes to indexes).
   - Highly configurable (`left_on`, `right_on`, `validate`, `indicator`).
2. **`df.join()`**:
   - High-level convenience method designed specifically for **joining on index labels**.
   - Defaults to `how='left'` (unlike `merge` which defaults to `how='inner'`).
   - Faster syntax when joining multiple DataFrames sharing identical indices (`df.join([df2, df3])`).

---

### Q2: How does Pandas perform joins internally, and how does key data type affect performance?
**Answer**:
- Pandas uses an optimized hash-table join (similar to SQL hash-join) implemented in Cython.
- **Key Dtype Trap**: If `left_on` is `int64` and `right_on` is `object` (string), `pd.merge()` will raise a `ValueError: You are trying to merge on int64 and object columns`.
- For categorical or low-cardinality string keys, converting keys to `'category'` or integer IDs before merging significantly cuts join memory and execution time.

---

### Q3: Advanced Interview Coding Drill: Cross-Week Customer Lifetime Spend
**Challenge**:
Calculate the **total dollar amount spent by each unique customer across both Week 1 and Week 2 combined**.
Display the **Top 5 highest spending customers** with their Full Name, Company, Occupation, and Total Spend!

In [8]:
# Interview Solution: Full Star-Schema Aggregation Pipeline
# 1. Combine week 1 and week 2 transactions
all_sales = pd.concat([week1, week2], ignore_index=True)

# 2. Attach food price (many-to-one)
all_sales_priced = all_sales.merge(
    foods["Price"],
    how="left",
    left_on="Food ID",
    right_index=True,
    validate="m:1"
)

# 3. Aggregate total spend per customer
customer_spend = (
    all_sales_priced.groupby("Customer ID")["Price"]
    .agg(total_spend="sum", total_orders="count")
    .reset_index()
)

# 4. Attach customer metadata (one-to-one join with customer dim)
top_spenders = (
    customer_spend.merge(
        customers[["First Name", "Last Name", "Company", "Occupation"]],
        how="inner",
        left_on="Customer ID",
        right_index=True,
        validate="1:1"
    )
    .sort_values(by="total_spend", ascending=False)
    .head(5)
)

print("Top 5 Customers by Combined Lifetime Spend (Week 1 + Week 2):")
display(top_spenders)

Top 5 Customers by Combined Lifetime Spend (Week 1 + Week 2):


,Customer ID,total_spend,total_orders,First Name,Last Name,Company,Occupation
31,77,40.95,5,Lori,Edwards,Dabvine,Sales Representative
139,343,40.46,4,Joe,Price,Fanoodle,Marketing Manager
225,550,38.98,2,Walter,Miller,Yata,Payment Adjustment Coordinator
102,253,38.73,3,Amy,Lawson,Browseblab,Compensation Analyst
208,520,36.24,2,Melissa,Wallace,Skyvu,Administrative Officer
